In [0]:
%run ../../config/config

In [0]:
%run  ../../config/sqlconfig

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, FloatType, TimestampType
from pyspark.sql.functions import col, lit, udf, to_timestamp, to_date, to_utc_timestamp, from_utc_timestamp, unix_timestamp, from_unixtime, datediff, months_between, round, dayofmonth, dayofweek, dayofyear, month, year, hour, minute, second, weekofyear, date_format, current_timestamp, current_date, date_add, date_sub, date_trunc, date_diff

In [0]:
class silverCustomerStreamETL:
    def __init__(self, spark):
        self.spark = spark
        self.catalog = CONFIG['catalog']['name']
        self.schema_name = CONFIG['catalog']['schema']
        
        # up-stream table name
        self.upstream_table_name = TABLES['bronze']['customer']

        # down-stream table name
        self.downstream_table_name = TABLES['silver']['customer']

        #path
        self.checkpoint_path = CONFIG['path']['checkpoint']

    def read_stream(self):
        df = self.spark.readStream.table(f"{self.catalog}.{self.schema_name}.{self.upstream_table_name }")
        return df

    def change_data_type(self, df):
        df_casted = (
            df.withColumn("customer_key", col("customer_key").cast(IntegerType()))
            .withColumn("customer_name", col("customer_name").cast(StringType()))
            .withColumn("gender", col("gender").cast(StringType()))
            .withColumn("city", col("city").cast(StringType()))
        )
        return df_casted

    def write_stream(self, df):
        try:
            query= (
                df.writeStream
                .option("checkpointLocation", f"{self.checkpoint_path}/{self.downstream_table_name}")
                .trigger(availableNow=True)
                .table(f"{self.catalog}.{self.schema_name}.{self.downstream_table_name}")
            )
            query.awaitTermination()
        except Exception as e:
            print(f"Error: write_stream {self.downstream_table_name}: {e}")

    def run(self):
        df = self.read_stream()
        df_casted = self.change_data_type(df)
        self.write_stream(df_casted)
        print("success full run")


In [0]:
obj = silverCustomerStreamETL(spark)
obj.run()

In [0]:
%sql
-- select *from bronze_customer